# Fatias da imagem e círculos da aorta

Analisa a extensão axial, a cobertura, o raio e a posição dos círculos da aorta nos conjuntos de treino e validação, relacionando essas medidas com a qualidade visual da máscara e o desfecho dos óstios.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import display

# Localiza a raiz antes dos imports internos quando o notebook abre em src/eda.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
    resolve_aorta_review_summary_path,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Configuração

As duas coortes usam o pipeline normal com limite inferior de -300 HU e limite superior P99.9. Os rótulos boa/ruim vêm da inspeção visual dos HTMLs 3D de cada conjunto.

In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
REVIEW_CATALOG = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
TRAIN_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "train")
VAL_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "val")
TRAIN_SUMMARY_PATH = resolve_aorta_review_summary_path(REPO_ROOT, TRAIN_REVIEW, "train")
VAL_SUMMARY_PATH = resolve_aorta_review_summary_path(REPO_ROOT, VAL_REVIEW, "val")

print(
    f"Treino: {len(TRAIN_REVIEW['aorta_good_ids'])} boas e "
    f"{len(TRAIN_REVIEW['aorta_bad_ids'])} ruins"
)
print(
    f"Validação: {len(VAL_REVIEW['aorta_good_ids'])} boas e "
    f"{len(VAL_REVIEW['aorta_bad_ids'])} ruins"
)

## 2. Carregamento e métricas derivadas

A cobertura é a fração das fatias da imagem que possuem um círculo final. As posições inicial, central e final são normalizadas pelo total de fatias para permitir comparar exames com extensões diferentes.

In [ ]:
CIRCLE_COLUMNS = [
    "IMG_ID", "image_slice_count", "aorta_circle_count",
    "aorta_detected_circle_count", "aorta_interpolated_circle_count",
    "aorta_circle_first_slice", "aorta_circle_last_slice",
    "aorta_circle_radius_mean_mm", "aorta_circle_radius_std_mm",
    "ostia_detection_status", "artery_dice",
]

def load_circle_cohort(summary_path, cohort_name, good_ids, bad_ids, bad_ostia_ids=None):
    """Carrega uma coorte e deriva medidas comparáveis entre exames."""
    cohort_df = pd.read_csv(summary_path)
    missing_columns = set(CIRCLE_COLUMNS).difference(cohort_df.columns)
    if missing_columns:
        raise ValueError(f"Colunas ausentes em {cohort_name}: {sorted(missing_columns)}")

    cohort_df = cohort_df[CIRCLE_COLUMNS].copy()
    for column in set(CIRCLE_COLUMNS).difference({"ostia_detection_status"}):
        cohort_df[column] = pd.to_numeric(cohort_df[column], errors="coerce")
    cohort_df["IMG_ID"] = cohort_df["IMG_ID"].astype(int)

    expected_ids = set(good_ids) | set(bad_ids)
    observed_ids = set(cohort_df["IMG_ID"])
    if expected_ids != observed_ids:
        raise ValueError(
            f"IDs incompatíveis em {cohort_name}. Ausentes={sorted(expected_ids - observed_ids)}; "
            f"não classificados={sorted(observed_ids - expected_ids)}"
        )

    # Normaliza extensão e posição pela quantidade total de fatias.
    cohort_df["circle_slice_fraction"] = cohort_df["aorta_circle_count"] / cohort_df["image_slice_count"]
    cohort_df["circle_first_position"] = cohort_df["aorta_circle_first_slice"] / cohort_df["image_slice_count"]
    cohort_df["circle_last_position"] = cohort_df["aorta_circle_last_slice"] / cohort_df["image_slice_count"]
    cohort_df["circle_center_position"] = (
        cohort_df[["aorta_circle_first_slice", "aorta_circle_last_slice"]].mean(axis=1)
        / cohort_df["image_slice_count"]
    )
    cohort_df["visual_aorta_quality"] = np.where(cohort_df["IMG_ID"].isin(good_ids), "boa", "ruim")
    normalized_status = cohort_df["ostia_detection_status"].astype(str).str.lower().str.replace("_", " ", regex=False)
    csv_ostia_success = normalized_status.isin(
        {"both correct", "both tolerable", "both ostia correct", "both ostia tolerable"}
    )
    # Em validação, prioriza a revisão visual fornecida; no treino, usa o status salvo.
    cohort_df["ostia_success"] = (
        ~cohort_df["IMG_ID"].isin(bad_ostia_ids)
        if bad_ostia_ids is not None else csv_ostia_success
    )
    cohort_df["ostia_outcome"] = np.where(cohort_df["ostia_success"], "sucesso", "falha")
    cohort_df["coorte"] = cohort_name
    return cohort_df.sort_values("IMG_ID").reset_index(drop=True)

train_df = load_circle_cohort(
    TRAIN_SUMMARY_PATH, "treino",
    TRAIN_REVIEW["aorta_good_ids"], TRAIN_REVIEW["aorta_bad_ids"],
    TRAIN_REVIEW["ostia_bad_ids"],
)
val_df = load_circle_cohort(
    VAL_SUMMARY_PATH, "validação",
    VAL_REVIEW["aorta_good_ids"], VAL_REVIEW["aorta_bad_ids"],
    VAL_REVIEW["ostia_bad_ids"],
)
print(f"Treino carregado: {len(train_df)} imagens")
print(f"Validação carregada: {len(val_df)} imagens")

## 3. Visão geral das coortes

In [ ]:
def summarize_cohort(cohort_df, cohort_name):
    """Resume a extensão e os círculos de uma coorte completa."""
    return {
        "coorte": cohort_name, "imagens": len(cohort_df),
        "fatias_imagem_media": cohort_df["image_slice_count"].mean(),
        "fatias_com_circulo_media": cohort_df["aorta_circle_count"].mean(),
        "cobertura_media_percentual": cohort_df["circle_slice_fraction"].mean() * 100,
        "raio_medio_mm": cohort_df["aorta_circle_radius_mean_mm"].mean(),
        "sucesso_ostios_percentual": cohort_df["ostia_success"].mean() * 100,
        "dice_arterial_medio": cohort_df["artery_dice"].mean(),
    }

overview_df = pd.DataFrame([summarize_cohort(train_df, "treino"), summarize_cohort(val_df, "validação")])
display(overview_df.round(3))

## 4. Círculos por qualidade visual da aorta

As mesmas métricas são mostradas primeiro para treino e depois para validação, facilitando comparar se o padrão se repete entre as coortes.

In [ ]:
def build_quality_summary(cohort_df):
    """Agrupa extensão, raio e posição pela avaliação visual."""
    return (
        cohort_df.groupby("visual_aorta_quality", observed=True)
        .agg(
            imagens=("IMG_ID", "size"), fatias_imagem_media=("image_slice_count", "mean"),
            fatias_com_circulo_media=("aorta_circle_count", "mean"),
            cobertura_media=("circle_slice_fraction", "mean"),
            raio_medio_mm=("aorta_circle_radius_mean_mm", "mean"),
            variacao_media_raio_mm=("aorta_circle_radius_std_mm", "mean"),
            primeira_posicao_media=("circle_first_position", "mean"),
            centro_posicao_media=("circle_center_position", "mean"),
            ultima_posicao_media=("circle_last_position", "mean"),
            sucesso_ostios=("ostia_success", "mean"), dice_medio=("artery_dice", "mean"),
        )
        .reset_index()
        .assign(
            cobertura_media=lambda frame: frame["cobertura_media"] * 100,
            primeira_posicao_media=lambda frame: frame["primeira_posicao_media"] * 100,
            centro_posicao_media=lambda frame: frame["centro_posicao_media"] * 100,
            ultima_posicao_media=lambda frame: frame["ultima_posicao_media"] * 100,
            sucesso_ostios=lambda frame: frame["sucesso_ostios"] * 100,
        )
    )

### 4.1. Treino

In [ ]:
display(build_quality_summary(train_df).round(3))

### 4.2. Validação

In [ ]:
display(build_quality_summary(val_df).round(3))

## 5. Relações por exame

A cor indica a qualidade da aorta; círculo indica sucesso dos óstios e `X` indica falha.

In [ ]:
QUALITY_COLORS = {"boa": "#2a9d8f", "ruim": "#d1495b"}
OUTCOME_MARKERS = {True: "o", False: "X"}

def plot_circle_relationships(cohort_df, cohort_name):
    """Plota cobertura, raio e intervalo axial de uma coorte."""
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
    for quality in ["boa", "ruim"]:
        for ostia_success in [True, False]:
            subset = cohort_df.loc[cohort_df["visual_aorta_quality"].eq(quality) & cohort_df["ostia_success"].eq(ostia_success)]
            style = {"color": QUALITY_COLORS[quality], "marker": OUTCOME_MARKERS[ostia_success], "s": 54, "alpha": 0.85, "edgecolor": "white", "linewidth": 0.6}
            axes[0].scatter(subset["image_slice_count"], subset["circle_slice_fraction"] * 100, **style)
            axes[1].scatter(subset["circle_center_position"] * 100, subset["aorta_circle_radius_mean_mm"], **style)
            axes[2].scatter(subset["circle_first_position"] * 100, subset["circle_last_position"] * 100, **style)
    axes[0].set(xlabel="Fatias da imagem", ylabel="Fatias com círculo (%)", title="Extensão e cobertura")
    axes[1].set(xlabel="Posição central (%)", ylabel="Raio médio (mm)", title="Posição e raio")
    axes[2].set(xlabel="Primeiro círculo (%)", ylabel="Último círculo (%)", title="Intervalo axial")
    for ax in axes:
        ax.grid(alpha=0.22)
    handles = [
        Patch(facecolor=QUALITY_COLORS["boa"], label="Aorta boa"), Patch(facecolor=QUALITY_COLORS["ruim"], label="Aorta ruim"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#555555", markersize=7, label="Óstios: sucesso"),
        Line2D([0], [0], marker="X", color="none", markerfacecolor="#555555", markersize=8, label="Óstios: falha"),
    ]
    fig.suptitle(cohort_name, y=1.02, fontsize=14)
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.99), ncols=4)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    plt.show()

### 5.1. Treino

In [ ]:
plot_circle_relationships(train_df, "Treino")

### 5.2. Validação

In [ ]:
plot_circle_relationships(val_df, "Validação")

## 6. Valores por exame

As tabelas começam pelas aortas ruins e, dentro de cada classe, pela menor cobertura.

In [ ]:
def build_case_table(cohort_df):
    """Seleciona as medidas essenciais para inspeção exame a exame."""
    return (
        cohort_df.assign(
            cobertura_percentual=cohort_df["circle_slice_fraction"] * 100,
            primeira_posicao_percentual=cohort_df["circle_first_position"] * 100,
            centro_posicao_percentual=cohort_df["circle_center_position"] * 100,
            ultima_posicao_percentual=cohort_df["circle_last_position"] * 100,
        )
        .sort_values(["visual_aorta_quality", "cobertura_percentual"], ascending=[False, True])
        [["IMG_ID", "visual_aorta_quality", "ostia_outcome", "image_slice_count", "aorta_circle_count", "aorta_detected_circle_count", "aorta_interpolated_circle_count", "cobertura_percentual", "aorta_circle_radius_mean_mm", "aorta_circle_radius_std_mm", "primeira_posicao_percentual", "centro_posicao_percentual", "ultima_posicao_percentual", "artery_dice"]]
    )

### 6.1. Treino

In [ ]:
display(build_case_table(train_df).round(3))

### 6.2. Validação

In [ ]:
display(build_case_table(val_df).round(3))

## 7. Leitura final

Compare primeiro as tabelas de qualidade visual e depois use os gráficos para verificar se as diferenças são gerais ou causadas por poucos exames extremos. As tabelas finais identificam diretamente esses casos em treino e validação.